# Week 03 Adapter Sprint: full QLoRA SFT run (Colab T4)

**Purpose:** run the **baseline** QLoRA instruction-tuning experiment once, on the exact commit reviewed for it. The resulting adapter is the starting point and reference for DPO.

All training logic lives in `training/train_qlora.py`. This notebook only calls repository scripts.

**Rules:**
- If any cell fails, **stop**. Do not change settings and re-run.
- Failed CUDA runs are logged automatically (OOM, non-finite loss, gradient-flow and other errors). Report the failure before any corrective run.
- No numbers are pre-filled. Everything shown comes from this execution.
- The smoke adapter in `outputs/smoke_adapter/` is **never** used here or for DPO.

## 1. Runtime / GPU check (before installing anything)

In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version.split()[0])
print("preinstalled torch", torch.__version__, "| built for CUDA", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("STOP: no CUDA GPU. Runtime > Change runtime type > T4 GPU, then restart from Section 1.")
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
with open("/content/torch_constraint.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")
print(open("/content/torch_constraint.txt").read())

## 2. Clone the repository
`EXPECTED_COMMIT` is **required**. Set it to the reviewed commit for this run (the short hash of `origin/main`). Preflight stops if HEAD differs.

In [ ]:
REPO_URL = "https://github.com/brentoatis-collab/week03-adapter-sprint.git"
REPO_DIR = "/content/week03-adapter-sprint"
EXPECTED_COMMIT = ""   # REQUIRED, e.g. "abc1234"

import os
if not EXPECTED_COMMIT:
    raise RuntimeError("STOP: set EXPECTED_COMMIT to the reviewed commit.")
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -5

## 3. Install pinned dependencies
torch stays pinned to Colab's preinstalled CUDA build through the constraint file.

A `pip check` line saying *ipython requires jedi* comes from Colab's own IPython and needs no action (see README §6). Any other `pip check` issue, especially one naming a pinned package, is a stop condition.

In [ ]:
%cd /content/week03-adapter-sprint
!pip install -r requirements.txt -c /content/torch_constraint.txt
!pip check || echo "pip check reported issues (see above) - review before continuing"

## 4. ⚠️ RESTART REQUIRED
**Runtime → Restart session**, then continue from **Section 5**. Do not re-run Sections 1–3.

## 5. Preflight on the exact commit
Checks:
- installed versions against the pins
- CUDA, `sm_75` kernels and compute capability
- fp16 matmul and the bitsandbytes NF4 + double-quant kernels
- clean tree, **HEAD == EXPECTED_COMMIT**
- frozen instruction dataset hashes

In [ ]:
import subprocess

def run(cmd):
    """Run a repository script, stream its output, and STOP the notebook on a non-zero exit."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"STOP: command failed with exit code {rc}: {' '.join(cmd)}")

import os, sys
os.chdir("/content/week03-adapter-sprint")
EXPECTED_COMMIT = ""   # REQUIRED again after the restart (same value as Section 2)
if not EXPECTED_COMMIT:
    raise RuntimeError("STOP: set EXPECTED_COMMIT.")
run([sys.executable, "training/preflight.py", "--require-cuda", "--expected-commit", EXPECTED_COMMIT])

## 6. Dataset integrity (no regeneration; frozen data must be unchanged)

In [ ]:
run([sys.executable, "scripts/audit_instruction_data.py"])
run(["git", "diff", "--exit-code", "--stat", "--", "data/"])
print("data/ matches the committed frozen datasets")

## 7. Masking sanity check + baseline dry run

In [ ]:
run([sys.executable, "training/check_masking.py"])
run([sys.executable, "training/train_qlora.py", "--run-label", "baseline", "--dry-run"])

## 8. FULL baseline QLoRA SFT run
This is the approved baseline, run with **no overrides**:
- `Qwen/Qwen2.5-0.5B-Instruct`, NF4 + double quantization, fp16 compute
- LoRA r=16, α=32, dropout 0.05 on q/k/v/o
- max sequence length 512, micro-batch 4 × accumulation 4 (effective 16)
- gradient checkpointing on
- LR 2e-4 with a cosine schedule, warmup ratio 0.1
- 3 epochs, max grad norm 0.3
- evaluation every 10 steps, plus one final evaluation of the final weights

Expected **66 optimizer steps**: 85 micro-batches per epoch → 21 full accumulation steps + 1 partial step (1 micro-batch) per epoch, × 3 epochs.

Output: `outputs/qlora_sft/qlora_sft-baseline-<timestamp>/final_adapter`, with `run_manifest.json` recording its SHA-256.

**If this cell fails: STOP.** The failure is already logged. Do not re-run with different settings; go to Section 10.

In [ ]:
run([sys.executable, "training/train_qlora.py", "--run-label", "baseline"])

## 9. Results: experiment-log row, memory phases, losses, adapter

In [ ]:
import glob, json, os
import pandas as pd

log = pd.read_csv("logs/experiment_log.csv")
base = log[log["run_label"] == "baseline"]
print(f"{len(log)} row(s) in log; {len(base)} baseline row(s)")
row = base.iloc[-1]
display(row.to_frame("value"))

mem_cols = ["peak_allocated_gb", "peak_reserved_gb",
            "peak_allocated_load_gb", "peak_reserved_load_gb",
            "peak_allocated_train_gb", "peak_reserved_train_gb",
            "peak_allocated_eval_gb", "peak_reserved_eval_gb",
            "alloc_retries", "cuda_ooms", "peak_inactive_split_gb", "peak_segments", "gpu_total_memory_gb"]
display(row[mem_cols].to_frame("memory"))

run_dir = sorted(glob.glob("outputs/qlora_sft/qlora_sft-baseline-*"))[-1]
manifest = json.load(open(os.path.join(run_dir, "run_manifest.json")))
print(json.dumps(manifest, indent=1))
hist = json.load(open(os.path.join(run_dir, "log_history.json")))
display(pd.DataFrame([{"step": h.get("step"), "loss": h.get("loss"), "grad_norm": h.get("grad_norm"),
                       "learning_rate": h.get("learning_rate"), "eval_loss": h.get("eval_loss")}
                      for h in hist if "loss" in h or "eval_loss" in h]))
print("\nSFT ADAPTER FOR DPO:", manifest["adapter_path"], "sha256", manifest["adapter_sha256"])
!git status --short
!git diff --stat

## 10. Persist results

Colab storage is temporary. Both of these are needed:

**(a) The experiment-log row.** Commit **only** `logs/experiment_log.csv` and push it, the same way as the smoke row. `git diff --stat` above should list only that file.

**(b) The SFT adapter.** DPO (Step 5B) will need it in a later session. The cell below copies the run directory to Google Drive and verifies the adapter SHA-256 after the copy.

In [ ]:
from google.colab import drive
import shutil, hashlib, os, json
drive.mount("/content/drive")
dest_root = "/content/drive/MyDrive/week03_adapter_sprint/qlora_sft"
os.makedirs(dest_root, exist_ok=True)
dest = os.path.join(dest_root, os.path.basename(run_dir))
if os.path.exists(dest):
    raise RuntimeError(f"STOP: {dest} already exists; not overwriting.")
shutil.copytree(run_dir, dest)
copied = os.path.join(dest, "final_adapter", "adapter_model.safetensors")
sha = hashlib.sha256(open(copied, "rb").read()).hexdigest()
print("copied to", dest)
print("sha256 after copy:", sha, "MATCH" if sha == manifest["adapter_sha256"] else "MISMATCH - STOP")
assert sha == manifest["adapter_sha256"]

### Run conclusion (fill in by hand from the output above; do not guess)

| Check | Evidence | Result |
|---|---|---|
| Preflight passed on EXPECTED_COMMIT | §5 | |
| Frozen data unchanged | §6 | |
| 66 optimizer steps completed (`total_steps`) | §9 | |
| Loss finite throughout; `GRADIENT CHECK` passed | §8 / §9 `notes` | |
| Final train / final eval loss | §9 | |
| Peak allocated / reserved (global and per phase) | §9 | |
| `alloc_retries`, `cuda_ooms`, `peak_inactive_split_gb` | §9 | |
| Wall clock / avg step time | §9 | |
| Provenance: python, torch CUDA build, GPU capability | §9 | |
| Adapter path + SHA-256; Drive copy verified | §9 / §10 | |
| Log row committed and pushed | §10 | |

**Interpretation reminders:**
- Allocated memory is tensor demand. Reserved memory is the allocator's cache. A reserved value near the device limit together with `alloc_retries > 0` indicates cache pressure, not demand above 15 GB.
- The eval loss is monitored only. The adapter is always the final step, because the eval set is also the held-out behavioral test set.

**If a step failed**, record:
- failure type and full error
- configuration
- likely failure class
- evidence
- **one** proposed corrective action

Do not retry until it has been reviewed.